# 🍔 FoodVision Big
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shalev396/ml-lab/blob/main/foodvision-big/training/notebook.ipynb)

Classify a food photo as one of the **101 dishes of Food-101**. An ImageNet-pretrained EfficientNet-B2
is fine-tuned end to end on all 75,750 training photos (Adam lr 1e-4, label smoothing 0.1,
TrivialAugmentWide, 5 epochs). This notebook drives every step (data, model, training, evaluation
against the deployed checkpoint, inference, export) by calling the functions in `src/`, and writes
the Hugging Face model folder that the [Space](https://huggingface.co/spaces/shalev396/foodvision-big) serves.

A full run needs a GPU (about 20-25 min on an RTX 2080 Ti for the original run; many hours on a CPU).
`SMOKE_TEST=1` runs the same steps on 64 + 64 streamed images in a few minutes on a CPU.

**Contents:** 1. Setup · 2. Config · 3. Data · 4. Load model · 5. Training · 6. Evaluation · 7. Inference · 8. Export

## 1. Setup
1a finds the project files (clones the repo on a fresh runtime), 1b installs the dependencies from the root `requirements.txt`, 1c picks the `device`.

In [ ]:
# 1a. Project + files — fetch the repo / model code only if they aren't here (e.g. a fresh Colab runtime)
import os, subprocess
from pathlib import Path

SLUG = "foodvision-big"
if not Path("src").is_dir():                      # not inside <slug>/training yet -> get the repo
    if not Path("ml-lab").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/shalev396/ml-lab.git"], check=True)
    os.chdir(f"ml-lab/{SLUG}/training")
if not Path("../model/model.py").exists():   # model/ is a git submodule (HF repo): code only, skip weights
    subprocess.run(["git", "submodule", "update", "--init", "--depth", "1", f"{SLUG}/model"],
                   cwd="../..", check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
print("working dir:", Path.cwd())

In [ ]:
# 1b. Dependencies — the root requirements.txt is the single source of truth for every project.
# Colab/Jupyter never install a repo's requirements on their own, so this cell does it (fast no-op when satisfied).
%pip install -q -r ../../requirements.txt

In [ ]:
# 1c. Device — whatever this machine has (cuda -> mps -> cpu); nothing platform-specific is installed.
import torch
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

## 2. Config
Every hyperparameter lives in `src/config.py`. The defaults are the original recipe: full fine-tune,
`Adam(lr=1e-4)`, cross-entropy with label smoothing 0.1, TrivialAugmentWide on the training images,
batch 32, 5 epochs, seed 42, keep the epoch with the lowest test loss. `SMOKE_TEST=1` shrinks the run
to a 64-image train slice + 64-image test slice read straight from the Hub parquet files and 1 epoch,
and exports to `outputs/smoke/model` instead of `../model`.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image as ShowImage, display
from PIL import Image

from src import data_setup, engine, export, model_builder, utils   # importing src puts ../model on sys.path
from src.config import Config
import model as M                                                    # ../model/model.py, what the Space runs

os.environ.setdefault("SMOKE_TEST", "0")   # "1" = quick smoke run; a SMOKE_TEST set outside the notebook wins
PUSH_TO_HUB = False                         # upload the exported model folder to the Hub (section 8)

cfg = Config()
ASSETS = cfg.run_dir / "assets"            # plots of this run; export copies them into model/assets
print(utils.lib_versions("torch", "torchvision", "datasets", "huggingface_hub", "numpy"))
print("export to:", cfg.model_dir)
cfg

## 3. Data
[Food-101](https://huggingface.co/datasets/ethz/food101): 101 dishes, 750 train and 250 test photos
each (the Hub calls the test split `validation`). A full run downloads the parquet shards once
(~5 GB) into `training/data/hf/`; the smoke run reads only a few 100-image row groups. If the Hub is
unreachable, `torchvision.datasets.Food101` is the fallback. Hub labels are remapped by name to
the alphabetical class order the model uses. Test images get `model.get_transform()` (resize 288
bicubic, center-crop 288, ImageNet mean/std); training images get TrivialAugmentWide first.

In [ ]:
train_loader, test_loader, class_names = data_setup.create_dataloaders(cfg, M.get_transform(), device)
train_ds, test_ds = train_loader.dataset, test_loader.dataset
counts = pd.DataFrame({"train": data_setup.class_counts(train_ds, class_names),
                       "test": data_setup.class_counts(test_ds, class_names)})
x, y = test_ds[0]
print(f"{len(class_names)} classes | {len(train_ds):,} train / {len(test_ds):,} test images")
print(f"sample tensor {tuple(x.shape)} {x.dtype}, label {class_names[y]!r}")
counts[(counts != 0).any(axis=1)].describe().round(1)   # images per class (classes present in this run)

In [ ]:
# A look at the data: raw photos (they vary in size and framing) and what TrivialAugmentWide does to one of them.
idx = np.linspace(0, len(test_ds) - 1, 8).astype(int)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, i in zip(axes.flat, idx):
    ax.imshow(test_ds.image(i))
    ax.set_title(class_names[test_ds.labels[i]].replace("_", " "))
    ax.axis("off")
plt.tight_layout()
plt.show()

augment = data_setup.train_transform(lambda img: img)      # augmentation only, no tensor conversion
fig, axes = plt.subplots(1, 5, figsize=(15, 3.2))
raw = train_ds.image(0)
for k, ax in enumerate(axes):
    ax.imshow(raw if k == 0 else augment(raw))
    ax.set_title("original" if k == 0 else f"TrivialAugmentWide #{k}")
    ax.axis("off")
plt.tight_layout()
plt.show()
pd.DataFrame([test_ds.image(i).size for i in idx], columns=["width", "height"]).describe().round(0)

## 4. Load model
`model_builder.build_model` creates `model.FoodVisionNet` (from `../model/model.py`) with ImageNet
weights and a new `Dropout(0.3) -> Linear(1408, 101)` head; the whole network trains.
`model_builder.load_deployed` loads the checkpoint that is deployed right now (from `../model`, or
from the Hub when the clone has no weights). It is evaluated next to the new run in section 6 and
stays deployed unless the new run beats it.

In [ ]:
net = model_builder.build_model(cfg, class_names).to(device)
train_net = model_builder.maybe_compile(net, cfg, device)   # torch.compile on CUDA if cfg.use_compile
print(net.backbone.classifier)
print(f"parameters: {utils.count_params(net):,} total, {model_builder.trainable_params(net):,} trainable")

In [ ]:
deployed_net, deployed_dir, deployed_info = model_builder.load_deployed(device)
print("deployed checkpoint:", deployed_info.get("variant", export.DEPLOYED_DEFAULT_NAME) if deployed_net else None)
print("provenance:", deployed_info.get("source"))

## 5. Training
Cross-entropy with label smoothing 0.1, Adam over every parameter, AMP on CUDA (`utils.autocast`).
The test split is evaluated after every epoch: Food-101 has no validation split, and the original
recipe keeps the epoch with the lowest test loss (in the original run that was the last one).
The original run took ~4 min per epoch (train + test) on an RTX 2080 Ti.

In [ ]:
loss_fn = engine.make_loss(cfg.label_smoothing)
optimizer = engine.make_optimizer(net, cfg.lr)
history, train_time_s, best_epoch = engine.train(train_net, train_loader, test_loader, loss_fn, optimizer,
                                                 cfg.epochs, device, keep=cfg.keep, log_every=cfg.log_every)
print(f"training time: {train_time_s:.0f} s, kept epoch {best_epoch}")

In [ ]:
curves = export.plot_history(history, ASSETS / "training_curves.png",
                             title=f"Retrain with training/ ({device}): train vs test per epoch")
display(ShowImage(filename=str(curves)))

## 6. Evaluation
Both variants are scored per image on the same test images: top-1 accuracy, top-5 accuracy and
macro F1. `export.select_best` keeps the deployed checkpoint unless the new run is strictly better
(top-1, then top-5).

In [ ]:
variants = []
if deployed_net is not None:
    result = engine.evaluate(deployed_net, test_loader, loss_fn, device)
    variants.append(export.deployed_variant(deployed_dir, deployed_info, result))
result = engine.evaluate(net, test_loader, loss_fn, device)
variants.append(export.new_run_variant(net, result, device, train_time_s))

best = export.select_best(variants)
print("best:", best.name)
pd.DataFrame(export.comparison(variants)).T.round(4)

In [ ]:
cm = export.plot_confusion_matrix(best.y_true, best.y_pred, class_names, ASSETS / "confusion_matrix.png",
                                  title=best.name)
display(ShowImage(filename=str(cm), width=560))
pc = export.plot_per_class(best.y_true, best.y_pred, class_names, ASSETS / "per_class_accuracy.png", title=best.name)
display(ShowImage(filename=str(pc), width=560))
if len(variants) > 1:
    chart = export.plot_comparison(export.comparison(variants), best.name, ASSETS / "comparison.png")
    display(ShowImage(filename=str(chart)))
export.top_confusions(best.y_true, best.y_pred, class_names, k=10)

## 7. Inference
Exactly what the Space and the Inference Endpoint run: the best weights are staged in a folder,
then `model.load(dir, device).predict(image, top_k=5)` returns the five most likely dishes.

In [ ]:
staged = export.save_weights(best, cfg.run_dir / "best")
predictor = M.load(staged, device)
examples = sorted((utils.SPACE_DIR / "examples").glob("*.jpg"))
fig, axes = plt.subplots(1, len(examples), figsize=(4 * len(examples), 4))
for ax, path in zip(np.atleast_1d(axes), examples):
    top5 = predictor.predict(path, top_k=5)
    top = next(iter(top5))
    print(path.name, {k: round(v, 4) for k, v in top5.items()})
    ax.imshow(Image.open(path))
    ax.set_title(f"{top.replace('_', ' ')} ({top5[top]:.1%})")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Export
Writes the best variant into `cfg.model_dir` (`../model`, or `outputs/smoke/model` for smoke runs):
`model.safetensors` + `config.json`, `metrics.json` with the comparison of every variant, the plots
in `assets/`, and the refreshed model card (metrics + Experiments tables). `model.py`, `handler.py`
and `requirements.txt` are already there. To publish, set `PUSH_TO_HUB = True` in section 2; the
token comes from `hf auth login` (local) or an `HF_TOKEN` secret / environment variable.
Smoke runs are never uploaded.

In [ ]:
metrics = export.build_metrics(
    best, variants,
    data={"n_train": len(train_ds), "n_test": len(test_ds), "n_classes": len(class_names)},
    params=utils.count_params(net), smoke=cfg.smoke,
    evaluation="per-image top-1 / top-5 / macro F1 on the test images of this run",
)
export.export(best, metrics, cfg.model_dir, ASSETS)

for path in sorted(cfg.model_dir.rglob("*")):
    if path.is_file() and not {"__pycache__", ".git"} & set(path.parts):
        print(f"{path.relative_to(cfg.model_dir).as_posix():32s} {path.stat().st_size / 1e6:8.2f} MB")

if PUSH_TO_HUB and not cfg.smoke:
    print(utils.upload_model(cfg.model_dir))